In [717]:
import pandas as pd
import numpy as np
from sklearn.utils import resample
from sklearn.preprocessing import StandardScaler, MinMaxScaler


In [718]:
# 데이터 로드
bmi_data_path = "data/bmi_data.csv" 
food_data_path = "data/food_data.csv"
exercise_data_path = "data/exercise_data.csv"

In [719]:
bmi_data = pd.read_csv(bmi_data_path)
food_data = pd.read_csv(food_data_path)
exercise_data = pd.read_csv(exercise_data_path)

In [720]:
bmi_data

,성별코드,연령대코드(5세단위),허리둘레,수축기혈압,이완기혈압,식전혈당(공복혈당),총콜레스테롤,HDL콜레스테롤,LDL콜레스테롤,트리글리세라이드,혈청지오티(AST),혈청지피티(ALT),감마지티피,흡연상태,음주여부,BMI
0,2,13,72.0,125.0,79.0,88.0,166.000000,98.000000,53.000000,75.0,24.0,14.0,12.0,1.0,0.0,22.892820
1,1,11,90.1,118.0,76.0,103.0,196.744106,57.035215,115.281692,105.0,54.0,63.0,51.0,3.0,0.0,22.857143
2,1,13,96.5,120.0,70.0,182.0,196.744106,57.035215,115.281692,105.0,24.0,35.0,71.0,3.0,1.0,27.681661
3,2,11,71.0,118.0,73.0,96.0,220.000000,72.000000,127.000000,105.0,31.0,17.0,16.0,1.0,0.0,20.811655
4,2,13,71.0,167.0,96.0,106.0,163.000000,68.000000,79.000000,79.0,30.0,18.0,14.0,1.0,0.0,20.811655
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,2,13,85.0,125.0,81.0,90.0,201.000000,56.000000,118.000000,135.0,26.0,25.0,11.0,1.0,0.0,25.390625
99996,2,13,79.0,135.0,74.0,101.0,196.744106,57.035215,115.281692,105.0,23.0,21.0,13.0,1.0,0.0,22.959184
99997,2,9,83.0,120.0,63.0,132.0,196.744106,57.035215,115.281692,105.0,48.0,38.0,11.0,1.0,1.0,33.333333
99998,2,8,65.5,100.0,61.0,88.0,196.744106,57.035215,115.281692,105.0,15.0,11.0,12.0,1.0,1.0,19.531250


In [721]:
food_data

,음식,칼로리,지방,포화지방,단일불포화지방,다중불포화지방,탄수화물,당류,단백질,식이섬유,콜레스테롤,나트륨,수분
0,cream cheese,51,5.0,2.900,1.300,0.200,0.8,0.500,0.9,0.0,14.6,0.016,7.6
1,neufchatel cheese,215,19.4,10.900,4.900,0.800,3.1,2.700,7.8,0.0,62.9,0.300,53.6
2,requeijao cremoso light catupiry,49,3.6,2.300,0.900,0.000,0.9,3.400,0.8,0.1,0.0,0.000,0.0
3,ricotta cheese,30,2.0,1.300,0.500,0.002,1.5,0.091,1.5,0.0,9.8,0.017,14.7
4,cream cheese low fat,30,2.3,1.400,0.600,0.042,1.2,0.900,1.2,0.0,8.1,0.046,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2390,muesli master crumble,124,2.4,1.000,19.800,8.500,4.5,0.042,2.4,0.0,0.0,0.000,0.0
2391,bran flakes,131,0.8,0.200,0.100,0.500,32.2,7.400,4.0,7.3,0.0,0.200,1.4
2392,nut cereal,245,5.9,1.000,3.900,0.500,46.2,17.200,4.0,3.6,0.0,0.300,2.6
2393,corn flakes,108,0.3,0.098,0.071,0.018,24.6,2.200,1.7,0.8,0.0,0.200,0.9


In [722]:
exercise_data

,운동명,단위체중당에너지소비량
0,바벨 스쿼트,6.0
1,다트,2.5
2,야구 캐치볼,2.5
3,당구,2.5
4,요가,2.5
...,...,...
370,새천년체조,3.5
371,훌라후프,4.0
372,번지피지오,5.0
373,wty운동,7.0


In [723]:
# 1️. 데이터 클리닝
## 불필요한 컬럼 제거 (예시: ID, 기타 불필요한 정보)
drop_columns = ["기타_불필요한_컬럼1", "기타_불필요한_컬럼2"]  # 실제 불필요한 컬럼을 여기에 추가
bmi_data.drop(columns=[col for col in drop_columns if col in bmi_data.columns], errors="ignore", inplace=True)

In [724]:
## 부적절한 레코드 제거
bmi_data = bmi_data[~bmi_data.isin(["부적합", "미평가"]).any(axis=1)]

In [725]:
## 비정상적인 데이터 형식 변환
bmi_data.replace({">40": "40", "60(80)": "70"}, inplace=True)

In [726]:
bmi_data['연령대코드(5세단위)'].unique()

array([13, 11,  7,  9, 12, 14, 10,  5,  8, 15,  6, 16, 17, 18])

In [727]:
# ✅ `user_id` 컬럼 확인 및 생성
if "user_id" not in bmi_data.columns:
    bmi_data["user_id"] = np.arange(0, len(bmi_data))
if "user_id" not in food_data.columns:
    food_data["user_id"] = np.random.choice(bmi_data["user_id"], size=len(food_data), replace=True)
if "user_id" not in exercise_data.columns:
    exercise_data["user_id"] = np.random.choice(bmi_data["user_id"], size=len(exercise_data), replace=True)


In [728]:
# ✅ `user_id` 개수 확인 (bmi_data 기준)
target_user_count = 99999
print(f" BMI 데이터 user_id 개수: {bmi_data['user_id'].nunique()}")
print(f" 운동 데이터 user_id 개수: {exercise_data['user_id'].nunique()}")
print(f" 음식 데이터 user_id 개수: {food_data['user_id'].nunique()}")

 BMI 데이터 user_id 개수: 100000
 운동 데이터 user_id 개수: 374
 음식 데이터 user_id 개수: 2369


In [729]:
# ✅ 오버샘플링 수행 함수


def oversample_data(df, target_count, label):
    """
    데이터프레임을 주어진 target_count에 맞춰 오버샘플링하는 함수
    """
    df = df.copy()  # 원본 데이터 수정 방지
    current_count = len(df)
    if current_count >= target_count:
        return df  # 이미 충분한 데이터가 있으면 유지

    # 오버샘플링 수행 (중복 허용)
    oversampled_df = resample(df,
                              replace=True,  # 중복 허용
                              n_samples=target_count,  # 목표 개수 맞추기
                              random_state=23)
    
    print(f"✅ {label} 데이터 오버샘플링 완료! 기존 {current_count} → {len(oversampled_df)} 개")
    return oversampled_df

In [731]:
# ✅ food_data 및 exercise_data 오버샘플링 (user_id = 99999에 맞춤)
target_user_count = 99999
food_data= oversample_data(food_data, target_user_count, "음식")
exercise_data = oversample_data(exercise_data, target_user_count, "운동명")


✅ 음식 데이터 오버샘플링 완료! 기존 2395 → 99999 개
✅ 운동명 데이터 오버샘플링 완료! 기존 375 → 99999 개


In [752]:
exercise_data

,운동명,단위체중당에너지소비량,user_id,운동 여부
83,스키,6.0,57646,1
230,허리돌리기(야외운동기구),2.5,53619,0
40,걷기,4.5,99485,1
31,골프,4.8,91291,1
237,배구연습(일반적인),4.0,83057,1
...,...,...,...,...
83,스키,6.0,57646,1
236,와이드 스쿼트,5.5,93307,1
134,고정식자전거타기(101~160 Watts),8.8,99030,1
326,크로스 크런치,4.5,99938,1


In [735]:
# 운동 여부를 판단하는 기준값 (예: MET 기준 설정)
ENERGY_THRESHOLD = 3.0  # ⚠️ 너무 높으면 모두 0이 될 수 있음, 적절한 값으로 조정 필요

In [736]:
# 🔹 운동 여부 추가 로직: '단위체중당에너지소비량'이 특정 임계값 이상이면 1, 아니면 0
exercise_data ["운동 여부"] = (exercise_data["단위체중당에너지소비량"] > ENERGY_THRESHOLD).astype(int)

In [754]:
exercise_data.isna().sum()


운동명            0
단위체중당에너지소비량    0
user_id        0
운동 여부          0
dtype: int64

In [737]:
# 운동 여부 값 확인
print(exercise_data["운동 여부"].value_counts())

운동 여부
1    88723
0    11276
Name: count, dtype: int64


In [738]:
unique_user_ids = bmi_data["user_id"].unique()
food_data["user_id"] = np.random.choice(unique_user_ids, size=len(food_data), replace=False)

In [739]:
food_data

,음식,칼로리,지방,포화지방,단일불포화지방,다중불포화지방,탄수화물,당류,단백질,식이섬유,콜레스테롤,나트륨,수분,user_id
595,pink lady apple,18,0.100,0.000,0.000,0.000,3.9,0.000,0.1,0.000,0.0,0.000,0.0,55932
742,garlic,4,0.026,0.033,0.078,0.089,0.9,0.043,0.2,0.064,0.0,0.018,1.6,50321
1064,pork tail cooked,1089,98.500,34.200,46.400,10.800,0.0,0.000,46.8,0.000,354.8,0.093,128.4,8587
1993,carrots raw,25,0.100,0.046,0.043,0.069,5.8,2.900,0.6,1.700,0.0,0.024,53.9,36310
1512,macaroni raw,390,1.600,0.300,0.200,0.600,78.4,2.800,13.7,3.400,0.0,0.097,10.4,3008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,sockeye salmon cooked,493,17.300,3.000,5.800,4.100,0.0,0.000,82.1,0.000,189.1,0.300,208.7,25589
8,parmesan cheese,71,4.500,2.700,1.400,0.100,0.6,0.046,6.4,0.000,12.2,0.200,5.4,5285
1765,blue cheese dressing light,14,0.400,0.100,0.200,0.100,2.1,0.600,0.3,0.000,1.6,0.200,12.4,52495
1130,whiskey sour,158,0.000,0.018,0.092,0.024,13.9,0.000,0.0,0.000,0.0,0.067,77.0,19573


In [740]:
# ✅ 섭취 여부 처리 (user_id 기준으로 칼로리 합산 후 1로 설정)
food_data_grouped = food_data.groupby("user_id", as_index=False)["칼로리"].sum()
food_data_grouped["섭취 여부"] = (food_data_grouped["칼로리"] > 0).astype(int)

In [741]:
food_data["섭취 여부"] = food_data_grouped["섭취 여부"] 


In [743]:
food_data

,음식,칼로리,지방,포화지방,단일불포화지방,다중불포화지방,탄수화물,당류,단백질,식이섬유,콜레스테롤,나트륨,수분,user_id,섭취 여부
595,pink lady apple,18,0.100,0.000,0.000,0.000,3.9,0.000,0.1,0.000,0.0,0.000,0.0,55932,1
742,garlic,4,0.026,0.033,0.078,0.089,0.9,0.043,0.2,0.064,0.0,0.018,1.6,50321,1
1064,pork tail cooked,1089,98.500,34.200,46.400,10.800,0.0,0.000,46.8,0.000,354.8,0.093,128.4,8587,1
1993,carrots raw,25,0.100,0.046,0.043,0.069,5.8,2.900,0.6,1.700,0.0,0.024,53.9,36310,1
1512,macaroni raw,390,1.600,0.300,0.200,0.600,78.4,2.800,13.7,3.400,0.0,0.097,10.4,3008,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
413,sockeye salmon cooked,493,17.300,3.000,5.800,4.100,0.0,0.000,82.1,0.000,189.1,0.300,208.7,25589,1
8,parmesan cheese,71,4.500,2.700,1.400,0.100,0.6,0.046,6.4,0.000,12.2,0.200,5.4,5285,1
1765,blue cheese dressing light,14,0.400,0.100,0.200,0.100,2.1,0.600,0.3,0.000,1.6,0.200,12.4,52495,1
1130,whiskey sour,158,0.000,0.018,0.092,0.024,13.9,0.000,0.0,0.000,0.0,0.067,77.0,19573,1


In [744]:
# 섭취 여부 값 확인
print(food_data_grouped["섭취 여부"].value_counts())

섭취 여부
1    99368
0      631
Name: count, dtype: int64


In [755]:
food_data.to_csv("./data/food_data_1.csv", index=False)
exercise_data.to_csv("./data/exercise_data_1.csv", index=False)
bmi_data.to_csv("./data/bmi_data_1.csv", index=False)


In [756]:
# 3. BMI 데이터와 병합 (균등 분포 보장)
merged_data = bmi_data.copy()
merged_data = merged_data.merge(exercise_data, on="user_id", how="left")
merged_data = merged_data.merge(food_data, on="user_id", how="left")

In [757]:
merged_data

,성별코드,연령대코드(5세단위),허리둘레,수축기혈압,이완기혈압,식전혈당(공복혈당),총콜레스테롤,HDL콜레스테롤,LDL콜레스테롤,트리글리세라이드,혈청지오티(AST),혈청지피티(ALT),감마지티피,흡연상태,음주여부,BMI,user_id,운동명,단위체중당에너지소비량,운동 여부
0,2,13,72.0,125.0,79.0,88.0,166.000000,98.000000,53.000000,75.0,24.0,14.0,12.0,1.0,0.0,22.892820,0,NaN,NaN,NaN
1,1,11,90.1,118.0,76.0,103.0,196.744106,57.035215,115.281692,105.0,54.0,63.0,51.0,3.0,0.0,22.857143,1,NaN,NaN,NaN
2,1,13,96.5,120.0,70.0,182.0,196.744106,57.035215,115.281692,105.0,24.0,35.0,71.0,3.0,1.0,27.681661,2,NaN,NaN,NaN
3,2,11,71.0,118.0,73.0,96.0,220.000000,72.000000,127.000000,105.0,31.0,17.0,16.0,1.0,0.0,20.811655,3,NaN,NaN,NaN
4,2,13,71.0,167.0,96.0,106.0,163.000000,68.000000,79.000000,79.0,30.0,18.0,14.0,1.0,0.0,20.811655,4,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199620,2,13,85.0,125.0,81.0,90.0,201.000000,56.000000,118.000000,135.0,26.0,25.0,11.0,1.0,0.0,25.390625,99995,NaN,NaN,NaN
199621,2,13,79.0,135.0,74.0,101.0,196.744106,57.035215,115.281692,105.0,23.0,21.0,13.0,1.0,0.0,22.959184,99996,NaN,NaN,NaN
199622,2,9,83.0,120.0,63.0,132.0,196.744106,57.035215,115.281692,105.0,48.0,38.0,11.0,1.0,1.0,33.333333,99997,NaN,NaN,NaN
199623,2,8,65.5,100.0,61.0,88.0,196.744106,57.035215,115.281692,105.0,15.0,11.0,12.0,1.0,1.0,19.531250,99998,NaN,NaN,NaN


In [747]:
# ✅ 결측값 처리
merged_data["운동 여부"] = merged_data["운동 여부"].fillna(0).astype(int)
merged_data["섭취 여부"] = merged_data["섭취 여부"].fillna(0).astype(int)

In [748]:
merged_data.isna().sum()

성별코드               0
연령대코드(5세단위)        0
허리둘레               0
수축기혈압              0
이완기혈압              0
식전혈당(공복혈당)         0
총콜레스테롤             0
HDL콜레스테롤           0
LDL콜레스테롤           0
트리글리세라이드           0
혈청지오티(AST)         0
혈청지피티(ALT)         0
감마지티피              0
흡연상태               0
음주여부               0
BMI                0
user_id            0
운동명            99626
단위체중당에너지소비량    99626
운동 여부              0
음식                 1
칼로리                1
지방                 1
포화지방               1
단일불포화지방            1
다중불포화지방            1
탄수화물               1
당류                 1
단백질                1
식이섬유               1
콜레스테롤              1
나트륨                1
수분                 1
섭취 여부              0
dtype: int64

In [651]:
print("🚀 최종 병합된 데이터 개수:", len(merged_data))

🚀 최종 병합된 데이터 개수: 199625


In [652]:
merged_data["운동 여부"].value_counts()

운동 여부
0    110902
1     88723
Name: count, dtype: int64

In [653]:
merged_data["섭취 여부"].value_counts()

섭취 여부
1    198136
0      1489
Name: count, dtype: int64

In [654]:
merged_data.columns

Index(['성별코드', '연령대코드(5세단위)', '허리둘레', '수축기혈압', '이완기혈압', '식전혈당(공복혈당)', '총콜레스테롤',
       'HDL콜레스테롤', 'LDL콜레스테롤', '트리글리세라이드', '혈청지오티(AST)', '혈청지피티(ALT)', '감마지티피',
       '흡연상태', '음주여부', 'BMI', 'user_id', '운동명', '단위체중당에너지소비량', '운동 여부', '음식',
       '칼로리', '지방', '포화지방', '단일불포화지방', '다중불포화지방', '탄수화물', '당류', '단백질', '식이섬유',
       '콜레스테롤', '나트륨', '수분', '섭취 여부'],
      dtype='object')

In [655]:
merged_data.to_csv("./data/merged_data.csv", index=False)

In [656]:
from sklearn.preprocessing import StandardScaler


In [657]:
# 연속형 변수 목록 중 실제 존재하는 컬럼만 필터링
continuous_columns = ["허리둘레", "수축기혈압", "이완기혈압", "식전혈당", "총콜레스테롤", "BMI"]
existing_continuous_columns = [col for col in continuous_columns if col in merged_data.columns]


In [658]:
# 정규화 수행
scaler = StandardScaler()
if existing_continuous_columns:
    merged_data[existing_continuous_columns] = scaler.fit_transform(merged_data[existing_continuous_columns])
    print("✅ 정규화 완료:", existing_continuous_columns)
else:
    print("🚨 정규화할 연속형 컬럼이 없습니다!")

✅ 정규화 완료: ['허리둘레', '수축기혈압', '이완기혈압', '총콜레스테롤', 'BMI']


In [659]:
# 4️.데이터 축소
## 중복 컬럼 처리 (평균값을 이용해 통합)
if "중복_컬럼1" in merged_data.columns and "중복_컬럼2" in merged_data.columns:
    merged_data["통합_컬럼"] = merged_data[["중복_컬럼1", "중복_컬럼2"]].mean(axis=1)
    merged_data.drop(columns=["중복_컬럼1", "중복_컬럼2"], inplace=True)

In [660]:
## 같은 날짜의 여러 측정값 평균 처리
if "검사날짜" in merged_data.columns:
    merged_data = merged_data.groupby(["user_id", "검사날짜"], as_index=False).mean()
    

In [661]:
# 5️. 데이터 이산화
## 연속형 변수를 범주화 (예: 크레아티닌 수치를 '정상', '경고', '위험'으로 구분)
def categorize_bmi(value):
    if value < 18.5:
        return "저체중"
    elif 18.5 <= value < 25:
        return "정상"
    elif 25 <= value < 30:
        return "과체중"
    else:
        return "비만"


In [662]:
merged_data["BMI_등급"] = merged_data["BMI"].apply(categorize_bmi)


In [663]:
print(merged_data.columns)

Index(['성별코드', '연령대코드(5세단위)', '허리둘레', '수축기혈압', '이완기혈압', '식전혈당(공복혈당)', '총콜레스테롤',
       'HDL콜레스테롤', 'LDL콜레스테롤', '트리글리세라이드', '혈청지오티(AST)', '혈청지피티(ALT)', '감마지티피',
       '흡연상태', '음주여부', 'BMI', 'user_id', '운동명', '단위체중당에너지소비량', '운동 여부', '음식',
       '칼로리', '지방', '포화지방', '단일불포화지방', '다중불포화지방', '탄수화물', '당류', '단백질', '식이섬유',
       '콜레스테롤', '나트륨', '수분', '섭취 여부', 'BMI_등급'],
      dtype='object')


In [664]:
merged_data

,성별코드,연령대코드(5세단위),허리둘레,수축기혈압,이완기혈압,식전혈당(공복혈당),총콜레스테롤,HDL콜레스테롤,LDL콜레스테롤,트리글리세라이드,...,다중불포화지방,탄수화물,당류,단백질,식이섬유,콜레스테롤,나트륨,수분,섭취 여부,BMI_등급
0,2,13,-0.891671,0.166165,0.373437,88.0,-1.301341,98.000000,53.000000,75.0,...,2.200,0.0,0.000,66.100,0.000,256.9,0.200,117.7,1,저체중
1,1,11,0.815206,-0.328938,0.074499,103.0,0.005489,57.035215,115.281692,105.0,...,0.100,0.0,0.000,5.200,0.000,15.6,0.200,22.8,1,저체중
2,1,13,1.418742,-0.187480,-0.523376,182.0,0.005489,57.035215,115.281692,105.0,...,1.100,110.3,1.200,15.500,14.900,0.0,0.062,17.9,1,저체중
3,2,11,-0.985973,-0.328938,-0.224439,96.0,0.994019,72.000000,127.000000,105.0,...,0.052,5.3,2.900,0.300,2.400,0.0,0.033,0.2,1,저체중
4,2,13,-0.985973,3.136785,2.067417,106.0,-1.428861,68.000000,79.000000,79.0,...,0.060,0.7,0.062,0.099,0.099,0.0,0.055,1.2,1,저체중
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199620,2,13,0.334262,0.166165,0.572728,90.0,0.186392,56.000000,118.000000,135.0,...,0.000,3.7,0.900,0.100,0.000,0.0,0.011,127.3,1,저체중
199621,2,13,-0.231553,0.873456,-0.124793,101.0,0.005489,57.035215,115.281692,105.0,...,0.200,16.4,9.500,2.200,4.300,0.0,0.049,219.7,1,저체중
199622,2,9,0.145657,-0.187480,-1.220898,132.0,0.005489,57.035215,115.281692,105.0,...,0.300,4.8,3.100,2.100,1.800,0.0,0.077,171.4,1,저체중
199623,2,8,-1.504638,-1.602061,-1.420190,88.0,0.005489,57.035215,115.281692,105.0,...,1.100,84.7,3.100,15.100,12.000,0.0,0.033,13.4,1,저체중


In [665]:
merged_data.columns

Index(['성별코드', '연령대코드(5세단위)', '허리둘레', '수축기혈압', '이완기혈압', '식전혈당(공복혈당)', '총콜레스테롤',
       'HDL콜레스테롤', 'LDL콜레스테롤', '트리글리세라이드', '혈청지오티(AST)', '혈청지피티(ALT)', '감마지티피',
       '흡연상태', '음주여부', 'BMI', 'user_id', '운동명', '단위체중당에너지소비량', '운동 여부', '음식',
       '칼로리', '지방', '포화지방', '단일불포화지방', '다중불포화지방', '탄수화물', '당류', '단백질', '식이섬유',
       '콜레스테롤', '나트륨', '수분', '섭취 여부', 'BMI_등급'],
      dtype='object')

In [666]:
merged_data.to_csv("./data/merged_data_1.csv", index=False)

In [716]:
merged_data.isna().sum()

성별코드               0
연령대코드(5세단위)        0
허리둘레               0
수축기혈압              0
이완기혈압              0
식전혈당(공복혈당)         0
총콜레스테롤             0
HDL콜레스테롤           0
LDL콜레스테롤           0
트리글리세라이드           0
혈청지오티(AST)         0
혈청지피티(ALT)         0
감마지티피              0
흡연상태               0
음주여부               0
BMI                0
user_id            0
운동명            99626
단위체중당에너지소비량    99626
운동 여부              0
음식                 1
칼로리                1
지방                 1
포화지방               1
단일불포화지방            1
다중불포화지방            1
탄수화물               1
당류                 1
단백질                1
식이섬유               1
콜레스테롤              1
나트륨                1
수분                 1
섭취 여부              0
BMI_등급             0
dtype: int64

In [669]:
merged_data['섭취 여부'].value_counts()


섭취 여부
1    198136
0      1489
Name: count, dtype: int64

In [698]:
merged_data['운동 여부'].value_counts()

운동 여부
0    110902
1     88723
Name: count, dtype: int64

In [670]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [671]:
# 🏷️ 사용할 특성 (X) 및 타겟 변수 (y)
features = [
    '성별코드', '연령대코드(5세단위)', '허리둘레', '수축기혈압', '이완기혈압',
    '식전혈당(공복혈당)', '총콜레스테롤', 'HDL콜레스테롤', 'LDL콜레스테롤', '트리글리세라이드',
    '혈청지오티(AST)', '혈청지피티(ALT)', '감마지티피', '흡연상태', '음주여부', 'BMI',
    '칼로리', '지방', '포화지방', '단일불포화지방', '다중불포화지방',
    '탄수화물', '당류', '단백질', '식이섬유', '콜레스테롤', '나트륨', '수분',
    '단위체중당에너지소비량'
]

# 🏋️‍♂️ 운동 여부 예측 데이터 준비
X_exercise = merged_data[features]
y_exercise = merged_data["운동 여부"]

# 🍽️ 음식 섭취 여부 예측 데이터 준비
X_food = merged_data[features]
y_food = merged_data["섭취 여부"]


In [672]:
# 결측값 제거
X_exercise = X_exercise.dropna()
X_food = X_food.dropna()

In [673]:
# 같은 행을 유지하기 위해 y도 정리
y_exercise = y_exercise.loc[X_exercise.index]
y_food = y_food.loc[X_food.index]

In [674]:
# 데이터 스케일링
scaler = StandardScaler()
X_exercise_scaled = scaler.fit_transform(X_exercise)
X_food_scaled = scaler.fit_transform(X_food)

In [675]:

# 데이터 분할 (훈련 80%, 테스트 20%)
X_train_ex, X_test_ex, y_train_ex, y_test_ex = train_test_split(X_exercise_scaled, y_exercise, test_size=0.2, random_state=23)
X_train_food, X_test_food, y_train_food, y_test_food = train_test_split(X_food_scaled, y_food, test_size=0.2, random_state=23)

In [676]:
# 음식 섭취 여부 예측 모델 학습
regressor_food = RandomForestClassifier(n_estimators=100, random_state=23)
regressor_food.fit(X_train_food, y_train_food)

RandomForestClassifier(random_state=23)

In [677]:
#  운동 여부 예측 모델 학습
regressor_exercise = RandomForestClassifier(n_estimators=100, random_state=23)
regressor_exercise.fit(X_train_ex, y_train_ex)

RandomForestClassifier(random_state=23)

In [678]:
# 음식 섭취 여부 예측
y_pred_food = regressor_food.predict(X_test_food)


In [679]:
accuracy_score(y_test_food, y_pred_food)

1.0

In [680]:
classification_report(y_test_food, y_pred_food)

'              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00       155\n           1       1.00      1.00      1.00     19845\n\n    accuracy                           1.00     20000\n   macro avg       1.00      1.00      1.00     20000\nweighted avg       1.00      1.00      1.00     20000\n'

In [681]:
#  운동 여부 예측
y_pred_exercise = regressor_exercise.predict(X_test_ex)

In [682]:
accuracy_score(y_test_ex, y_pred_exercise)

1.0

In [683]:
classification_report(y_test_ex, y_pred_exercise)

'              precision    recall  f1-score   support\n\n           0       1.00      1.00      1.00      2277\n           1       1.00      1.00      1.00     17723\n\n    accuracy                           1.00     20000\n   macro avg       1.00      1.00      1.00     20000\nweighted avg       1.00      1.00      1.00     20000\n'

In [684]:
import joblib
import pickle
import os

In [685]:
with open("data/regressor_exercise.pkl", "wb") as f:
    pickle.dump(regressor_exercise, f)


In [686]:

with open("data/regressor_food.pkl", "wb") as f:
    pickle.dump(regressor_food, f)

In [687]:
#  운동 모델 로드
with open("data/regressor_exercise.pkl", "rb") as f: 
    regressor_exercise = pickle.load(f)


In [688]:
with open("data/regressor_food.pkl", "rb") as f: 
    regressor_food = pickle.load(f)

In [689]:
# 예측 예제 (운동 여부)
sample_data_dict = {
    "성별코드": [1], "연령대코드(5세단위)": [35], "허리둘레": [85], "BMI": [23.5], "칼로리": [2000],
    "총콜레스테롤": [190], "수축기혈압": [120], "이완기혈압": [80], "식전혈당(공복혈당)": [90], "음주여부": [1],
    "흡연상태": [0], "HDL콜레스테롤": [50], "LDL콜레스테롤": [100], "트리글리세라이드": [150], "혈청지오티(AST)": [25],
    "혈청지피티(ALT)": [30], "감마지티피": [35], "지방": [70], "포화지방": [20], "단일불포화지방": [25], 
    "다중불포화지방": [15], "탄수화물": [300], "당류": [50], "단백질": [100], "식이섬유": [30], 
    "콜레스테롤": [180], "나트륨": [1500], "수분": [2500], "운동명": [1]  # 운동명 (0=운동 안함, 1=운동함)
}


In [690]:
sample_data = pd.DataFrame(sample_data_dict)

In [691]:
#  학습 당시 사용한 StandardScaler 적용 (저장된 scaler 사용 필요)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(sample_data)

In [692]:
prediction = regressor_exercise.predict(X_scaled)

In [693]:
prediction

array([1])

In [694]:
# 🏷️ 사용할 특성 (X) 및 타겟 변수 (y)
features = [
    '성별코드', '연령대코드(5세단위)', '허리둘레', '수축기혈압', '이완기혈압',
    '식전혈당(공복혈당)', '총콜레스테롤', 'HDL콜레스테롤', 'LDL콜레스테롤', '트리글리세라이드',
    '혈청지오티(AST)', '혈청지피티(ALT)', '감마지티피', '흡연상태', '음주여부', 'BMI',
    '칼로리', '지방', '포화지방', '단일불포화지방', '다중불포화지방',
    '탄수화물', '당류', '단백질', '식이섬유', '콜레스테롤', '나트륨', '수분',
    '단위체중당에너지소비량'
]

In [695]:
#  모델 학습 시 사용된 피처 목록을 저장할 파일 경로
FEATURES_FILE = "data/feature_columns.pkl"

In [696]:
# ✅ 디렉토리 존재 여부 확인 후 생성
os.makedirs(os.path.dirname(FEATURES_FILE), exist_ok=True)

# ✅ 피처 목록 저장
with open(FEATURES_FILE, "wb") as f:
    pickle.dump(features, f)

In [697]:
# 저장된 피처 목록 불러오기
with open("data/feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

In [701]:
# XGBoost 및 Deep Learning(딥러닝) 학습 코드

In [713]:
from imblearn.over_sampling import SMOTE
import xgboost as xgb
from sklearn.model_selection import train_test_split

In [703]:
X = merged_data[features]
y_food = merged_data["섭취 여부"]
y_exercise = merged_data["운동 여부"]

In [706]:
# ✅ 데이터 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [708]:
# ✅ 섭취 여부 예측 (심각한 불균형 → SMOTE 적용)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_food, test_size=0.2, random_state=23, stratify=y_food)

In [709]:
food_smote = SMOTE(random_state=23)

In [715]:
food_smote.fit_resample(X_train, y_train)

ValueError: Input X contains NaN.
SMOTE does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values